In [ ]:
#! pip install ultralytics
#! pip install opencv-python
#! pip install numpy

In [26]:
import cv2
from ultralytics import YOLO

# -------------------------
# CONFIG
# -------------------------

VIDEO_PATH = "input/footage_2.mp4"
MODEL_PATH = "yolov8n.pt"

# focal length stimata (pixel)
FOCAL_LENGTH = 960

# dimensioni reali medie (metri)
REAL_HEIGHTS = {
    "car": 1.5,
    "person": 1.7,
    "bicycle": 1.2,
    "truck": 3.0,
    "bus": 3.2,
    "motorbike": 1.4
}

# soglia collision warning (metri)
COLLISION_THRESHOLD = 12

TARGET_CLASSES = set(REAL_HEIGHTS.keys())

print(TARGET_CLASSES)

{'motorbike', 'car', 'bus', 'truck', 'person', 'bicycle'}


In [27]:
# -------------------------
# DISTANCE FUNCTION
# -------------------------
def estimate_distance(label, pixel_height):
    if pixel_height <= 0:
        return None
    if label not in REAL_HEIGHTS:
        return None

    real_h = REAL_HEIGHTS[label]
    return (real_h * FOCAL_LENGTH) / pixel_height

In [28]:
# -------------------------
# LOAD MODEL + VIDEO
# -------------------------
model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise IOError("Cannot open video")


In [24]:
# -------------------------
# MAIN LOOP
# -------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]

    for box in results.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]

        if label not in TARGET_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        pixel_height = y2 - y1

        distance = estimate_distance(label, pixel_height)
        if distance is None:
            continue

        warning = distance < COLLISION_THRESHOLD
        color = (0,0,255) if warning else (0,255,0)

        # bounding box
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

        # label text
        text = f"{label} {distance:.1f}m"
        cv2.putText(frame, text, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # warning text
        if warning:
            cv2.putText(frame,
                        "FORWARD COLLISION WARNING",
                        (50,70),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (0,0,255),
                        3)

    cv2.imshow("ADAS Perception Demo", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break


cap.release()
cv2.destroyAllWindows()

In [29]:
# -------------------------
# PROCESS & SAVE VIDEO 
# -------------------------
def process_and_save_video(input_path, output_path):
    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        raise IOError("Cannot open video")

    # Video properties
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)

    # Define codec and writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)[0]

        for box in results.boxes:
            cls_id = int(box.cls[0])
            label = model.names[cls_id]

            if label not in TARGET_CLASSES:
                continue

            conf = float(box.conf[0])
            if conf < 0.5:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            pixel_height = y2 - y1

            distance = estimate_distance(label, pixel_height)
            if distance is None:
                continue

            warning = distance < COLLISION_THRESHOLD
            color = (0,0,255) if warning else (0,255,0)

            # draw bounding box
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

            text = f"{label} {distance:.1f}m"
            cv2.putText(frame, text, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            if warning:
                cv2.putText(frame,
                            "FORWARD COLLISION WARNING",
                            (50,70),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,
                            (0,0,255),
                            3)

        out.write(frame)

    cap.release()
    out.release()
    print(f"Saved processed video to {output_path}")

process_and_save_video("input/footage_2.mp4", "output/footage_2_out.mp4")

Saved processed video to output/footage_2_out.mp4
